# Feature maps with entanglement
**Quantum Machine Learning · Week 1 (Data meets qubits) · Day 4**

Lesson page: https://sara-aminpour.com/tutorials/qml/w1d4.html

In [ ]:
# On Google Colab, this installs qiskit (about a minute). On your own computer, skip it and
# install the packages once instead: pip install qiskit==2.5.2 scikit-learn torch matplotlib
import sys
if "google.colab" in sys.modules:
    %pip install --quiet qiskit==2.5.2

Angle encoding treats every feature separately. A **feature map with entanglement** lets features
interact, like the product feature $x_1x_2$ in ML week 4. The best-known one is the **ZZ feature map**.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit.circuit.library import zz_feature_map
from qiskit.quantum_info import Statevector, state_fidelity, partial_trace

def zz_map(x, reps=2):
    qc = QuantumCircuit(2)
    for _ in range(reps):
        qc.h([0, 1])
        qc.p(2 * x[0], 0)
        qc.p(2 * x[1], 1)
        qc.cx(0, 1)
        qc.p(2 * (np.pi - x[0]) * (np.pi - x[1]), 1)     # a phase that depends on BOTH features
        qc.cx(0, 1)
    return qc

x = np.array([0.7, 2.1])
library = zz_feature_map(2, reps=2).assign_parameters(x)      # Qiskit's version
print("same state as Qiskit's zz_feature_map:", np.isclose(state_fidelity(Statevector(zz_map(x)), Statevector(library)), 1))
zz_map(x, reps=1).draw("mpl", style="clifford")

## Entangled or not?
Angle encoding always gives product states. The ZZ map usually entangles: the purity of one qubit on its
own drops below 1 (QC week 6).

In [ ]:
def angle_map(x):
    qc = QuantumCircuit(2)
    qc.ry(x[0], 0)
    qc.ry(x[1], 1)
    return qc

rng = np.random.default_rng(0)
for name, fmap in [("angle", angle_map), ("ZZ", zz_map)]:
    purities = [partial_trace(Statevector(fmap(rng.uniform(0, np.pi, 2))), [1]).purity().real for _ in range(200)]
    print(f"{name:5s}: purity of qubit 0 alone, over 200 random inputs: min {min(purities):.3f}, mean {np.mean(purities):.3f}")

## Similarity landscapes
Fix a reference point and color every other input by the kernel $k(x, x_{\text{ref}}) = |\langle\varphi(x)|\varphi(x_{\text{ref}})\rangle|^2$.

In [ ]:
grid = np.linspace(0, np.pi, 70)
ref = np.array([0.9, 2.2])
fig, axes = plt.subplots(1, 3, figsize=(10, 3.3), layout="constrained")
for ax, (name, fmap) in zip(axes, [("angle", angle_map), ("ZZ, 1 layer", lambda x: zz_map(x, 1)), ("ZZ, 2 layers", zz_map)]):
    r = Statevector(fmap(ref))
    K = np.array([[abs(Statevector(fmap([a, b])).inner(r)) ** 2 for a in grid] for b in grid])
    ax.imshow(K, origin="lower", extent=[0, np.pi, 0, np.pi], cmap="Purples", vmin=0, vmax=1)
    ax.plot(*ref, "*", ms=14, color="#ffd66b", mec="#2c2842")
    ax.set_title(name)
    ax.set_xlabel(r"$x_1$")
    ax.grid(False)
axes[0].set_ylabel(r"$x_2$")
plt.show()

## Your turn: is more entanglement better?
An XOR pattern in the square $[0, \pi]^2$: class 1 where exactly one feature is above $\pi/2$. Build the kernel
matrix for each feature map (angle, ZZ with 1 layer, ZZ with 2 layers), train `SVC(kernel="precomputed", C=10)`
and compare test accuracies.

In [ ]:
from sklearn.svm import SVC

def xor_data(n, seed):
    r = np.random.default_rng(seed)
    X = r.uniform(0, np.pi, (n, 2))
    return X, ((X[:, 0] > np.pi / 2) ^ (X[:, 1] > np.pi / 2)).astype(int)

X_train, y_train = xor_data(80, 1)
X_test, y_test = xor_data(200, 2)

def kernel(fmap, A, B):
    SA = np.array([Statevector(fmap(a)).data for a in A])
    SB = np.array([Statevector(fmap(b)).data for b in B])
    return np.abs(SA.conj() @ SB.T) ** 2

for name, fmap in [("angle", angle_map), ("ZZ, 1 layer", lambda x: zz_map(x, 1)), ("ZZ, 2 layers", zz_map)]:
    pass    # TODO: fit SVC(kernel="precomputed", C=10) on kernel(fmap, X_train, X_train); score on kernel(fmap, X_test, X_train)

---
## Solution
Try the exercise yourself first: it's fine to get stuck for a minute. Then compare with this one (there are many good answers).

In [ ]:
for name, fmap in [("angle", angle_map), ("ZZ, 1 layer", lambda x: zz_map(x, 1)), ("ZZ, 2 layers", zz_map)]:
    model = SVC(kernel="precomputed", C=10).fit(kernel(fmap, X_train, X_train), y_train)
    print(f"{name:13s} test accuracy {model.score(kernel(fmap, X_test, X_train), y_test):.3f}")
print("More entanglement isn't automatically better: the two-layer ZZ map's similarity is so wiggly that")
print("the SVM generalizes worse here. A feature map has to suit the data.")